<a href="https://colab.research.google.com/github/cmjw/NASA-LOFT-State-Classification/blob/main/group_subject_weighting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = "/content/drive/My Drive/Thesis/Datasets/NASA_LOFT/Data"

zip_files = [
    f for f in os.listdir(DATA_DIR)
    if f.endswith(".zip")
]
print(f"Files: {zip_files}")

subjects = sorted(
    int(os.path.splitext(f)[0])
    for f in zip_files
)
print(f"Subjects: {subjects}")

import zipfile

for subject in subjects:
  zip_path = os.path.join(DATA_DIR, f"{subject}.zip")
  with zipfile.ZipFile(zip_path) as z:
    print(z.namelist())


import pandas as pd
import numpy as np

Mounted at /content/drive
Files: ['13.zip', '11.zip', '25.zip', '17.zip', '18.zip', '19.zip', '12.zip', '24.zip', '21.zip', '9.zip', '14.zip', '23.zip', '10.zip', '22.zip', '5.zip', '6.zip']
Subjects: [5, 6, 9, 10, 11, 12, 13, 14, 17, 18, 19, 21, 22, 23, 24, 25]
['5/', '5/5_DA.csv', '5/5_LOFT.csv', '5/5_CA.csv', '5/5_SS.csv']
['6/', '6/6_SS.csv', '6/6_DA.csv', '6/6_LOFT.csv', '6/6_CA.csv']
['9/', '9/9_DA.csv', '9/9_LOFT.csv', '9/9_CA.csv', '9/9_SS.csv']
['10/', '10/10_DA.csv', '10/10_LOFT.csv', '10/10_CA.csv', '10/10_SS.csv']
['11/', '11/11_LOFT.csv', '11/11_SS.csv', '11/11_DA.csv', '11/11_CA.csv']
['12/', '12/12_DA.csv', '12/12_CA.csv', '12/12_LOFT.csv', '12/12_SS.csv']
['13/', '13/13_SS.csv', '13/13_LOFT.csv', '13/13_DA.csv', '13/13_CA.csv']
['14/', '14/14_CA.csv', '14/14_DA.csv', '14/14_SS.csv', '14/14_LOFT.csv']
['17/', '17/17_SS.csv', '17/17_LOFT.csv', '17/17_CA.csv', '17/17_DA.csv']
['18/', '18/18_CA.csv', '18/18_DA.csv', '18/18_SS.csv', '18/18_LOFT.csv']
['19/', '19/19_SS.csv', 

In [14]:
# helper functions

# classify between CA, DA, and SS. There are so many more samples of baseline
# that we want to classify between events rather than event vs. no event.

def extract_non_baseline_events(df):
  df = df.copy()

  # treat event 3 or 4 as 0
  df["Event"] = df["Event"].replace({3: 0, 4: 0})

  # keep only non-baseline (1, 2, 5)
  df = df[df["Event"].isin([1, 2, 5])].copy()

  df["State"] = df["Event"].map({
      1: "SS",
      2: "CA",
      5: "DA"
  })
  return df

def extract_events(df):
  df = df.copy()

  # treat event 3 or 4 as 0
  df["Event"] = df["Event"].replace({3: 0, 4: 0})

  df = df[df["Event"].isin([0, 1, 2, 5])].copy()

  df["State"] = df["Event"].map({
      0: "BL",
      1: "SS",
      2: "CA",
      5: "DA"
  })
  return df

def find_event_intervals(df):
  intervals = []

  current_event = df["Event"].iloc[0]
  start_time = df["TimeSecs"].iloc[0]

  for i in range(1, len(df)):
      event = df["Event"].iloc[i]

      if event != current_event:
          end_time = df["TimeSecs"].iloc[i - 1]

          intervals.append({
              "Event": current_event,
              "Start": start_time,
              "End": end_time
          })

          current_event = event
          start_time = df["TimeSecs"].iloc[i]

  # Add the final interval
  intervals.append({
      "Event": current_event,
      "Start": start_time,
      "End": df["TimeSecs"].iloc[-1]
  })

  return pd.DataFrame(intervals)

import neurokit2 as nk

# feature extraction
def extract_features_per_window(df, window_size=10, stride=5, fs=256):
  '''
  Extract features over a window_size second window, for now only GSR/Resp
  '''
  features = []
  start = df["TimeSecs"].min()
  end_time = df["TimeSecs"].max()

  while start + window_size <= end_time:
    end = start + window_size

    window = df[(df["TimeSecs"] >= start) & (df["TimeSecs"] < end)].copy()

    if len(window) == 0:
      start += stride
      continue

    ecg_data = window["ECG"]
    peaks, info = nk.ecg_peaks(ecg_data, sampling_rate=fs)

    features.append({
        "GSR_mean": window["GSR"].mean(),
        "GSR_std":  window["GSR"].std(),
        "GSR_min":  window["GSR"].min(),
        "GSR_max":  window["GSR"].max(),
        "R_mean":   window["R"].mean(),
        "R_std":    window["R"].std(),
        "R_min":    window["R"].min(),
        "R_max":    window["R"].max(),
        "Event":    window["Event"].mode()[0]
    })
  return pd.DataFrame(features)

def extract_features_per_interval(df, window_size=10):
  '''
  Extract features for each event interval, of window size window_size
  '''
  intervals = find_event_intervals(df)
  features = []

  # extract features for each event
  for i, interval in intervals.iterrows():
    interval_df = df[(df["TimeSecs"] >= interval["Start"]) &
                     (df["TimeSecs"] <= interval["End"])]
    if len(interval_df) == 0:
      continue
    feats = extract_features_per_window(interval_df, window_size=window_size)
    #print(f"Event {interval_df["Event"].iloc[0]}: {len(feats)} features")
    features.append(feats)

  return pd.concat(features, ignore_index=True)

In [3]:
all_features = []

for subject in [5, 6, 9, 10, 11, 12, 13, 14, 17]:
  print(f"\nProcessing subject {subject}")

  zip_path = os.path.join(DATA_DIR, f"{subject}.zip")
  with zipfile.ZipFile(zip_path) as z:
    data = {
        "LOFT": ("flight_sim", pd.read_csv(z.open(f"{subject}/{subject}_LOFT.csv"))),
        "DA":   ("benchmark",  pd.read_csv(z.open(f"{subject}/{subject}_DA.csv"))),
        "CA":   ("benchmark",  pd.read_csv(z.open(f"{subject}/{subject}_CA.csv"))),
        "SS":   ("benchmark",  pd.read_csv(z.open(f"{subject}/{subject}_SS.csv")))
    }

    for session, (environment, df) in data.items():
      print(f"{session}: shape = {df.shape}")
      features = extract_features_per_interval(df, window_size=10)
      features["Subject"] = subject
      features["Session"] = session
      features["Environment"] = environment
      print(f"{session} features: shape = {features.shape}")
      all_features.append(features)

all_features = pd.concat(all_features, ignore_index=True)
print(f"All features: shape = {all_features.shape}")
print(all_features.head())


Processing subject 5
LOFT: shape = (1037788, 26)
LOFT features: shape = (413, 12)
DA: shape = (92077, 26)
DA features: shape = (47, 12)
CA: shape = (92131, 26)
CA features: shape = (37, 12)
SS: shape = (39563, 26)
SS features: shape = (17, 12)

Processing subject 6
LOFT: shape = (1037807, 26)
LOFT features: shape = (413, 12)
DA: shape = (92130, 26)
DA features: shape = (45, 12)
CA: shape = (92168, 26)
CA features: shape = (37, 12)
SS: shape = (39583, 26)
SS features: shape = (17, 12)

Processing subject 9
LOFT: shape = (1258929, 26)
LOFT features: shape = (500, 12)
DA: shape = (92194, 26)
DA features: shape = (47, 12)
CA: shape = (92133, 26)
CA features: shape = (37, 12)
SS: shape = (92131, 26)
SS features: shape = (39, 12)

Processing subject 10
LOFT: shape = (1257953, 27)
LOFT features: shape = (500, 12)
DA: shape = (92099, 27)
DA features: shape = (46, 12)
CA: shape = (92099, 27)
CA features: shape = (37, 12)
SS: shape = (92212, 27)
SS features: shape = (39, 12)

Processing subject

In [4]:
print(all_features["Event"].value_counts().sort_index())

Event
0.0    3940
1.0      64
2.0     679
5.0     158
Name: count, dtype: int64


In [12]:
# random forest models
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import balanced_accuracy_score

# prepare training data
target_subject = 6

train = all_features[all_features["Subject"] != target_subject]
test  = all_features[all_features["Subject"] == target_subject]
test  = test[test["Session"] == "LOFT"]

# remove event 0 class
train = train[train["Event"] != 0.0]
test  = test[test["Event"] != 0.0]

X_train = train.drop(columns=["Event", "Subject", "Session", "Environment"])
y_train = train["Event"]

X_test = test.drop(columns=["Event", "Subject", "Session", "Environment"])
y_test = test["Event"]

# model
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))
print("Balanced accuracy:",
      balanced_accuracy_score(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

         1.0       0.00      0.00      0.00         4
         2.0       0.57      0.09      0.15        45
         5.0       0.44      0.50      0.47         8

    accuracy                           0.14        57
   macro avg       0.34      0.20      0.21        57
weighted avg       0.51      0.14      0.19        57

Balanced accuracy: 0.1962962962962963
Confusion matrix:
[[ 0  2  2]
 [38  4  3]
 [ 3  1  4]]


SVM models

Class imbalance
- Removing event 0 causing prediction to collapse to the next most numerous class 2.
- Using class_weight="balanced" reverses this to predict the least numerous class, class 1.

In [8]:
# SVM models

# prepare training data
target_subject = 6

train = all_features[all_features["Subject"] != target_subject]
test  = all_features[all_features["Subject"] == target_subject]
test  = test[test["Session"] == "LOFT"]

# remove event 0 class
train = train[train["Event"] != 0.0]
test  = test[test["Event"] != 0.0]

X_train = train.drop(columns=["Event", "Subject", "Session", "Environment"])
y_train = train["Event"]

X_test = test.drop(columns=["Event", "Subject", "Session", "Environment"])
y_test = test["Event"]

# normalize training data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# train group model with sample left out
from sklearn.svm import SVC

group_model = SVC(kernel="rbf") # , class_weight="balanced"
group_model.fit(X_train, y_train)

# prediction
group_pred = group_model.predict(X_test)

# train subject-weighted model
sample_weights = np.ones(len(train))

subject_mask = (
    (train["Subject"] == target_subject) &
    (train["Environment"] == "benchmark")
)

sample_weights[subject_mask] = 5

weighted_model = SVC(kernel="rbf") # , class_weight="balanced"

weighted_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

weighted_pred = weighted_model.predict(X_test)

# results
from sklearn.metrics import classification_report, balanced_accuracy_score

print("GROUP MODEL")
print(classification_report(y_test, group_pred))
print("Balanced accuracy:",
      balanced_accuracy_score(y_test, group_pred))

print("\nWEIGHTED MODEL")
print(classification_report(y_test, weighted_pred))
print("Balanced accuracy:",
      balanced_accuracy_score(y_test, weighted_pred))

from sklearn.metrics import confusion_matrix

print("Group:")
print(confusion_matrix(y_test, group_pred, labels=[0, 1, 2, 5]))

print("\nWeighted:")
print(confusion_matrix(y_test, weighted_pred, labels=[0, 1, 2, 5]))